In [2]:
import numpy as np
import pandas as pd
import plotly as plt
import mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import math
import warnings
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.graph_objects as go
import plotly.express as px
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
df = pd.read_csv('Market_Basket_Optimisation.csv', header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       7501 non-null   object
 1   1       5747 non-null   object
 2   2       4389 non-null   object
 3   3       3345 non-null   object
 4   4       2529 non-null   object
 5   5       1864 non-null   object
 6   6       1369 non-null   object
 7   7       981 non-null    object
 8   8       654 non-null    object
 9   9       395 non-null    object
 10  10      256 non-null    object
 11  11      154 non-null    object
 12  12      87 non-null     object
 13  13      47 non-null     object
 14  14      25 non-null     object
 15  15      8 non-null      object
 16  16      4 non-null      object
 17  17      4 non-null      object
 18  18      3 non-null      object
 19  19      1 non-null      object
dtypes: object(20)
memory usage: 1.1+ MB


In [5]:
transactions = [row.dropna().tolist() for index, row in df.iterrows()]

In [6]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

df_onehot = pd.DataFrame(te_ary, columns=te.columns_)
df_onehot.head()

,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,True,True,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


In [7]:
df_onehot = df_onehot.loc[:, ~df_onehot.columns.duplicated()]

In [8]:
df_onehot.head()

,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,True,True,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


In [9]:
# Apriori with 1% support
frequent_items= apriori(df_onehot, min_support=0.01, use_colnames=True)
print(frequent_items)

      support                                 itemsets
0    0.020397                                (almonds)
1    0.033329                                (avocado)
2    0.010799                         (barbecue sauce)
3    0.014265                              (black tea)
4    0.011465                             (body spray)
..        ...                                      ...
252  0.011065       (milk, mineral water, ground beef)
253  0.017064  (mineral water, ground beef, spaghetti)
254  0.015731         (milk, mineral water, spaghetti)
255  0.010265    (olive oil, mineral water, spaghetti)
256  0.011465     (mineral water, pancakes, spaghetti)

[257 rows x 2 columns]


In [10]:
#Lift
rules = association_rules(frequent_items, metric="lift", min_threshold=1.2)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(avocado),(mineral water),0.033329,0.238368,0.011598,0.348000,1.459926,1.0,0.003654,1.168147,0.325896,0.044593,0.143943,0.198329
1,(mineral water),(avocado),0.238368,0.033329,0.011598,0.048658,1.459926,1.0,0.003654,1.016113,0.413630,0.044593,0.015857,0.198329
2,(cake),(burgers),0.081056,0.087188,0.011465,0.141447,1.622319,1.0,0.004398,1.063198,0.417434,0.073129,0.059442,0.136473
3,(burgers),(cake),0.087188,0.081056,0.011465,0.131498,1.622319,1.0,0.004398,1.058080,0.420238,0.073129,0.054892,0.136473
4,(eggs),(burgers),0.179709,0.087188,0.028796,0.160237,1.837830,1.0,0.013128,1.086988,0.555754,0.120941,0.080026,0.245256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,"(mineral water, spaghetti)",(pancakes),0.059725,0.095054,0.011465,0.191964,2.019529,1.0,0.005788,1.119933,0.536902,0.080000,0.107090,0.156291
344,"(spaghetti, pancakes)",(mineral water),0.025197,0.238368,0.011465,0.455026,1.908923,1.0,0.005459,1.397557,0.488452,0.045479,0.284466,0.251562
345,(mineral water),"(spaghetti, pancakes)",0.238368,0.025197,0.011465,0.048098,1.908923,1.0,0.005459,1.024059,0.625163,0.045479,0.023494,0.251562
346,(pancakes),"(mineral water, spaghetti)",0.095054,0.059725,0.011465,0.120617,2.019529,1.0,0.005788,1.069244,0.557862,0.080000,0.064760,0.156291


In [11]:
#Filter rules using 30% confidence
filtered_rules = rules[rules['confidence'] >= 0.3]
filtered_rules.sort_values('lift',ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
159,(herb & pepper),(ground beef),0.049460,0.098254,0.015998,0.323450,3.291994,1.0,0.011138,1.332860,0.732460,0.121457,0.249734,0.243136
324,"(mineral water, ground beef)",(spaghetti),0.040928,0.174110,0.017064,0.416938,2.394681,1.0,0.009938,1.416470,0.607262,0.086195,0.294020,0.257474
308,"(mineral water, frozen vegetables)",(milk),0.035729,0.129583,0.011065,0.309701,2.389991,1.0,0.006435,1.260929,0.603138,0.071737,0.206934,0.197546
195,(soup),(milk),0.050527,0.129583,0.015198,0.300792,2.321232,1.0,0.008651,1.244861,0.599484,0.092158,0.196697,0.209038
171,(ground beef),(spaghetti),0.098254,0.174110,0.039195,0.398915,2.291162,1.0,0.022088,1.373997,0.624943,0.168096,0.272197,0.312015


In [12]:
# Only rule requirements for this project: antecedents, consequents, support, confidence, lift
final_rules = filtered_rules[['antecedents','consequents','support','confidence','lift']]
final_rules.sort_values('lift',ascending=False)

,antecedents,consequents,support,confidence,lift
159,(herb & pepper),(ground beef),0.015998,0.323450,3.291994
324,"(mineral water, ground beef)",(spaghetti),0.017064,0.416938,2.394681
308,"(mineral water, frozen vegetables)",(milk),0.011065,0.309701,2.389991
195,(soup),(milk),0.015198,0.300792,2.321232
171,(ground beef),(spaghetti),0.039195,0.398915,2.291162
...,...,...,...,...,...
123,(frozen smoothie),(mineral water),0.020264,0.320000,1.342461
179,(honey),(mineral water),0.015065,0.317416,1.331619
185,(low fat yogurt),(mineral water),0.023997,0.313589,1.315565
117,(fresh bread),(mineral water),0.013332,0.309598,1.298820


In [13]:
#Creating DiGraph
G = nx.DiGraph()

In [14]:
final_rules.loc[:, 'antecedents'] = final_rules['antecedents'].apply(lambda x: ', '.join(x))
final_rules.loc[:, 'consequents'] = final_rules['consequents'].apply(lambda x: ', '.join(x))

In [15]:
for _, row in final_rules.iterrows():
  G.add_edge(
      row['antecedents'],
      row['consequents'],
      weight=row['lift'],
      support=row['support'],
      confidence=row['confidence']


  )

pos = nx.spring_layout(G, weight='weight', seed=42)

In [16]:
print(list(G.nodes())[:5])
print(list(G.edges())[:5])

['avocado', 'mineral water', 'burgers', 'eggs', 'cake']
[('avocado', 'mineral water'), ('burgers', 'eggs'), ('cake', 'mineral water'), ('cereals', 'mineral water'), ('chicken', 'mineral water')]


In [17]:
print(pos['mineral water'])

[0.10842319 0.08143401]


In [18]:
print(final_rules['antecedents'].head())

0     avocado
5     burgers
32       cake
39    cereals
50    chicken
Name: antecedents, dtype: object


In [19]:
print(final_rules['consequents'].head())

0     mineral water
5              eggs
32    mineral water
39    mineral water
50    mineral water
Name: consequents, dtype: object


In [20]:
list('avacado')

['a', 'v', 'a', 'c', 'a', 'd', 'o']

In [21]:
#Plotly graphs have two traces
#Trace 1
edge_x = []
edge_y = []
for u,v in G.edges():
  x0, y0 = pos[u]
  x1, y1 = pos[v]
  edge_x.extend([x0, x1, None])
  edge_y.extend([y0, y1, None])
edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=0.5, color='#888'), hoverinfo='none')


In [22]:
#Trace 2
node_x = [pos[node][0] for node in G.nodes()]
node_y = [pos[node][1] for node in G.nodes()]

node_degrees = dict(G.degree())
node_degree_values = [node_degrees[node] *5 for node in G.nodes()]

node_trace = go.Scatter(x=node_x, y=node_y,
                        mode='markers',
                        hoverinfo='text',
                        text=list(G.nodes()),
                        textposition='middle center',
                        marker=dict(
                            line=dict(width=0.5, color='#888'),
                            showscale=True,
                            colorscale='YlGnBu',
                            size=node_degree_values)
)


In [23]:
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(showlegend=False, xaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

In [24]:
print(type(fig))

<class 'plotly.graph_objs._figure.Figure'>


In [28]:
# Product Placement Recommendation
# Print 10 final consequents with highest lift in decending order
final_rules.sort_values('lift',ascending=False).head(10).reset_index(drop=True)

,antecedents,consequents,support,confidence,lift
0,herb & pepper,ground beef,0.015998,0.323450,3.291994
1,"mineral water, ground beef",spaghetti,0.017064,0.416938,2.394681
2,"mineral water, frozen vegetables",milk,0.011065,0.309701,2.389991
3,soup,milk,0.015198,0.300792,2.321232
4,ground beef,spaghetti,0.039195,0.398915,2.291162
5,"olive oil, mineral water",spaghetti,0.010265,0.371981,2.136468
6,"eggs, ground beef",mineral water,0.010132,0.506667,2.125563
7,"milk, ground beef",mineral water,0.011065,0.503030,2.110308
8,red wine,spaghetti,0.010265,0.364929,2.095966
9,olive oil,spaghetti,0.022930,0.348178,1.999758


In [42]:
# Promotional Branding Recommendation
# Top 5 most popular products
print("These 5 items can be bundled with 5 underperforming items for stock clearance")
frequent_items.sort_values('support',ascending=False).head(5).reset_index(drop=True)

These 5 items can be bundled with 5 underperforming items for stock clearance


,support,itemsets
0,0.238368,(mineral water)
1,0.179709,(eggs)
2,0.174110,(spaghetti)
3,0.170911,(french fries)
4,0.163845,(chocolate)


In [43]:
# 5 most underperforming items

#Filter single items only
single_items = frequent_items[frequent_items['itemsets'].apply(lambda x: len(x) == 1)]

# sort and take bottom 5 performing items
print("These 5 items can be bundled with 5 most popular items for stock clearance")
single_items.sort_values('support', ascending=True).head(5).reset_index(drop=True)

These 5 items can be bundled with 5 most popular items for stock clearance


,support,itemsets
0,0.010399,(nonfat milk)
1,0.010532,(cider)
2,0.010799,(barbecue sauce)
3,0.010932,(magazines)
4,0.011465,(body spray)
